In [1]:
import requests
import pandas as pd

c:\Users\Sultan Selin\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
cities = {
    "Istanbul": (41.01, 28.97),
    "Ankara": (39.93, 32.86),
    "Izmir": (38.42, 27.14)
}
weather_data = {}

url = "https://archive-api.open-meteo.com/v1/archive"

for city, coordinates in cities.items():
    latitude, longitude = coordinates

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2017-01-01",
        "end_date": "2026-08-30",
        "hourly": "temperature_2m",
        "timezone": "Europe/Istanbul"
    }

    response = requests.get(url, params=params)
    data = response.json()
    city_df = pd.DataFrame(data["hourly"])
    weather_data[city] = city_df

    print("Response_code:", response.status_code)
    print("Json_Keys:", response.json().keys())

In [25]:
print( weather_data["Ankara"].info())
print( weather_data["Izmir"].info())
print( weather_data["Istanbul"].info())

<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   time    84696 non-null  datetime64[us]
 1   Ankara  84696 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.3 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   time    84696 non-null  datetime64[us]
 1   Izmir   84696 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.3 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   time      84696 non-null  datetime64[us]
 1   Istanbul  84696 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory 

In [21]:
print( weather_data["Ankara"].describe())
print( weather_data["Izmir"].describe())
print( weather_data["Istanbul"].describe())

       temperature_2m
count    84696.000000
mean        13.120112
std          9.751840
min        -14.400000
25%          5.300000
50%         12.800000
75%         20.400000
max         40.500000
       temperature_2m
count    84696.000000
mean        18.839826
std          8.531541
min         -3.600000
25%         12.200000
50%         18.100000
75%         25.300000
max         43.100000
       temperature_2m
count    84696.000000
mean        15.438413
std          7.624451
min         -6.600000
25%          9.300000
50%         15.100000
75%         21.600000
max         39.600000


In [24]:

for city, city_df in weather_data.items():
    city_df["time"] = pd.to_datetime(city_df["time"])
    weather_data[city] = city_df.rename(
    columns={"temperature_2m": city}
    )

In [26]:
weather_data["Ankara"].head()

,time,Ankara
0,2017-01-01 00:00:00,-5.3
1,2017-01-01 01:00:00,-6.6
2,2017-01-01 02:00:00,-7.6
3,2017-01-01 03:00:00,-7.9
4,2017-01-01 04:00:00,-7.2


In [32]:
weather_df = pd.merge(
    weather_data["Istanbul"],
    weather_data["Ankara"],
    on="time"
)
weather_df = pd.merge(
    weather_df,
    weather_data["Izmir"],
    on="time"
)

In [ ]:
print(weather_df.head())
print(weather_df.info())


                 time  Istanbul  Ankara  Izmir
0 2017-01-01 00:00:00       4.0    -5.3    0.6
1 2017-01-01 01:00:00       3.7    -6.6   -0.1
2 2017-01-01 02:00:00       3.5    -7.6   -0.5
3 2017-01-01 03:00:00       1.6    -7.9    0.2
4 2017-01-01 04:00:00       1.5    -7.2    0.3
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   time      84696 non-null  datetime64[us]
 1   Istanbul  84696 non-null  float64       
 2   Ankara    84696 non-null  float64       
 3   Izmir     84696 non-null  float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 2.6 MB
None


In [36]:
populations = {
    "Istanbul": 15754053,
    "Ankara": 5910320,
    "Izmir": 4504185
}
total_population = sum(populations.values())

In [38]:
weights = {
    city: population / total_population
    for city, population in populations.items()
}
weights

{'Istanbul': 0.6020222054268333,
 'Ankara': 0.22585577699772375,
 'Izmir': 0.17212201757544301}

In [41]:
weather_df["national_temp"] = weather_df["Istanbul"]*weights["Istanbul"]+weather_df["Ankara"]*weights["Ankara"]+weather_df["Izmir"]*weights["Izmir"]
weather_df.head()

,time,Istanbul,Ankara,Izmir,national_temp
0,2017-01-01 00:00:00,4.0,-5.3,0.6,1.314326
1,2017-01-01 01:00:00,3.7,-6.6,-0.1,0.719622
2,2017-01-01 02:00:00,3.5,-7.6,-0.5,0.304513
3,2017-01-01 03:00:00,1.6,-7.9,0.2,-0.786601
4,2017-01-01 04:00:00,1.5,-7.2,0.3,-0.671492


In [42]:
from pathlib import Path
WEATHER_PATH = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "weather_data.parquet"
)
weather_df.to_parquet(WEATHER_PATH, index=False)